<a href="https://colab.research.google.com/github/adizquier/03MIAR_Algoritmos-de-Optimizacion/blob/main/Trabajo%20Practico/Trabajo_Pr%C3%A1ctico_Adrian_Izquierdo_Abril.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Adrián Izquierdo Abril  <br>
Url: https://github.com/adizquier/03MIAR_Algoritmos-de-Optimizacion/tree/main/Trabajo%20Practico<br>
Google Colab: https://colab.research.google.com/drive/1OXeJzjwtmkZv1Z8tEsEBvs4h2wvilw1c?usp=sharing <br>
Problema:
>1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de una jornada de La Liga<br>
>3. Configuración de Tribunales

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los
datos son:

1. Número de actores: 10
2. Número de tomas : 30
3. Actores/Tomas : https://bit.ly/36D8IuK

                                        

In [1]:
#Importaciones y rutas
import sys
import math
import time
import numpy as np
import pandas as pd

RUTA_EXCEL = "Datos problema doblaje(30 tomas, 10 actores).xlsx"

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

Los datos se proveen desde un archivo *.xlsx*, en el cual el valor de las filas viene dado por el número de tomas que hay (30 en total) mas una fila adicional que indica el número total de tomas que realiza un actor, y en las columnas viene dado el número de actores mas una columna adicional que indica el total de actores por toma. El contenido de cada celda viene dado por un valor binario (0 y 1) indicando si un actor tiene que asistir o no a una toma.

Con esta información, es posible modelar el problema y establecer las estructuras de datos necesarias para resolver el problema:

1. ¿Como se representa el espacio de soluciones?

    El espacio de soluciones es el conjunto de todas las asignaciones posibles de las 30 tomas a grupos (días) que cumplen las restricciones del problema. Cada elemento de ese espacio es una partición de las tomas en días, donde el orden de los días no importa. Este espacio crece exponencialmente con el número de tomas, lo que justifica el uso de heurísticas o de búsqueda con poda en lugar de una exploración exhaustiva. Una solución concreta vendrá representada por vectores que indiquen qué tomas se realizan en cada día, así como el total de actores que por tanto tendrán que asistir en función de las tomas seleccionadas.

2. ¿Cual es la función objetivo?

    La función objetivo resulta en minimizar el coste total, consiguiendo que los actores tengan que ir el menor número de veces posible a grabar. El coste de cada día viene dado por el número de actores únicos que hay que convocar, es decir, el total de actores distintos que aparecen en alguna de las tomas grabadas ese día. La función objetivo se expresa como la suma, sobre todos los días de grabación, del número de actores distintos convocados cada día.

3. ¿Cómo implemento las restricciones?

    El modelo presenta tres restricciones:

   - **Cobertura completa**: cada toma debe grabarse exactamente una vez, sin omisiones ni repeticiones.
   - **Capacidad máxima por día**: el número de tomas que se puede realizar por día es como máximo 6. Con 30 tomas y este límite, el número mínimo de días necesarios es 5.
   - **Convocatoria obligatoria**: si una toma se asigna a un día determinado y un actor aparece en ella, dicho actor debe ser convocado ese día.

La carga de datos se hará utilizando la librería *pandas*, especialmente el método *read_excel* para cargar el archivo de los datos. La fila de etiquetas, la fila de cabecera y la fila de totales no aportan información relevante para la resolución del problema, por lo que estas se obvian.
   
El modelado de los datos de entrada se realizará mediante una matriz de tamaño *numero_tomas X numero_actores*, con el contenido binario que corresponde a si un actor está o no en dicha toma. Esta construcción, pese a que los datos son bien conocidos, se realiza en tiempo de ejecución calculando sus valores dinámicamente.

La restricción en el número de tomas que se puede realizar por día, puesto que con los datos que se dispone se conoce que es seis, se define mediante una constante.

In [2]:
#Carga de datos
MAX_TOMAS_DIA = 6

df_raw = pd.read_excel(RUTA_EXCEL, header=None)
cabecera   = df_raw.iloc[1]


cols_actores = [
    c for c in df_raw.columns
    if c != 0
    and pd.notna(cabecera[c])
    and str(cabecera[c]).strip().replace(".0", "").isdigit()
]

n_actores = len(cols_actores)

filas_tomas = []
for i in range(2, len(df_raw)):
    val = df_raw.iloc[i, 0]
    try:
        num = int(float(val))
        filas_tomas.append(i)

    except (ValueError, TypeError):
        pass

n_tomas = len(filas_tomas)

matriz = np.zeros((n_tomas, n_actores), dtype=int)

for fila_idx, excel_fila in enumerate(filas_tomas):
    for col_idx, excel_col in enumerate(cols_actores):
        val = df_raw.iloc[excel_fila, excel_col]
        matriz[fila_idx, col_idx] = int(float(val)) if pd.notna(val) else 0

print(f"Datos cargados correctamente:")
print(f"Tomas     : {n_tomas}")
print(f"Actores   : {n_actores}")

Datos cargados correctamente:
Tomas     : 30
Actores   : 10


#Diseño
- ¿Que técnica utilizo? ¿Por qué?

Las dos técnicas que se implementarán son: Algoritmo voraz y Ramificación y Poda.

La elección del algoritmo voraz se ha tomado en base a que en este problema, la elección de un óptimo local lleva a pensar en la obtención de un resultado cercano al óptimo global. Si en el problema se seleccionan en primer lugar aquellas tomas que impliquen el mayor número de actores y, para las siguientes 5 tomas restantes (restricción de 6 tomas por día) se toman aquellas tomas con un mayor número de actores en común con la ya tomada, se puede pensar que tras los 5 días mínimos se obtendrá una planificación razonable de las tomas asignadas para cada día.

No obstante, esta solución no asegura un óptimo global, pues es posible que la elección de la primera toma del día, la cual se hará en base al número de actores por toma, haga que las demás tomas del día tengan pocos actores en común con ella, obteniendo un coste cercano al verdadero óptimo global, pero no necesariamente igual.

Es por esto que, sin esperar el mejor resultado posible, se prueba el diseño de un algoritmo voraz para la implementación del problema. Además, este tipo de algoritmos obtienen buenos resultados en bajos tiempos de ejecución.

La elección de la primera toma del día se hará con aquella que implique el mayor número de actores, como ya se ha comentado, y se seguirán rellenando las tomas restantes del día en base al número de actores compartidos con las ya seleccionadas, es decir, eligiendo en cada paso la toma que incorpore el menor número de actores nuevos.

In [3]:
def coste_sesion(indices_tomas, matriz, n_actores) -> int:
    """Número de actores únicos necesarios para un conjunto de tomas en el mismo día."""
    if not indices_tomas:
        return 0
    union = np.zeros(n_actores, dtype=int)
    for t in indices_tomas:
        union = np.maximum(union, matriz[t])
    return int(union.sum())


def actores_en_sesion(indices_tomas, matriz, n_actores) -> list:
    """Lista (1-based) de actores necesarios para un conjunto de tomas."""
    union = np.zeros(n_actores, dtype=int)
    for t in indices_tomas:
        union = np.maximum(union, matriz[t])
    return [a + 1 for a in range(n_actores) if union[a]]

In [4]:
def algoritmo_voraz(matriz, n_tomas, n_actores, max_tomas_dia) -> tuple:
    """Método principal que soluciona el problema con un algoritmo voraz."""

    tomas_pendientes = list(range(n_tomas))
    planificacion    = []

    while tomas_pendientes:

        # Toma con más actores
        toma_ancla = max(tomas_pendientes, key=lambda t: int(matriz[t].sum()))

        sesion_hoy  = [toma_ancla]
        actores_hoy = matriz[toma_ancla].copy().astype(int)
        tomas_pendientes.remove(toma_ancla)

        while len(sesion_hoy) < max_tomas_dia and tomas_pendientes:
            mejor_toma       = None
            menor_incremento = float('inf')

            for t in tomas_pendientes:
                nueva_union = np.maximum(actores_hoy, matriz[t])
                incremento  = int(nueva_union.sum()) - int(actores_hoy.sum())
                if incremento < menor_incremento:
                    menor_incremento = incremento
                    mejor_toma = t

            sesion_hoy.append(mejor_toma)
            actores_hoy = np.maximum(actores_hoy, matriz[mejor_toma])
            tomas_pendientes.remove(mejor_toma)

        planificacion.append(sesion_hoy)

    coste_total = sum(coste_sesion(s, matriz, n_actores) for s in planificacion)

    return planificacion, coste_total

In [5]:
def mostrar_resultado(planificacion, coste):
    """Método que se utilizará para representar las soluciones."""

    print(f"  Días necesarios : {len(planificacion)}")
    print(f"\n  {'DÍA':<5} {'TOMAS GRABADAS':<28} {'ACTORES CONVOCADOS':<26} {'COSTE'}")
    print("  " + "-" * 68)

    for i, sesion in enumerate(planificacion):
        tomas_str   = str(sorted(t + 1 for t in sesion))
        actores     = actores_en_sesion(sesion, matriz, n_actores)
        actores_str = str(actores)
        coste_dia   = len(actores)
        print(f"  {i+1:<5} {tomas_str:<28} {actores_str:<26} {coste_dia}")

    print(f"\n  COSTE TOTAL: {coste} actor-días")

In [6]:
start_time = time.time()
planificacionVoraz, costeVoraz = algoritmo_voraz(matriz, n_tomas, n_actores, MAX_TOMAS_DIA)
end_time = time.time()

mostrar_resultado(planificacionVoraz, costeVoraz)
print(f"  Tiempo de ejecución: {end_time - start_time} segundos")

  Días necesarios : 5

  DÍA   TOMAS GRABADAS               ACTORES CONVOCADOS         COSTE
  --------------------------------------------------------------------
  1     [1, 2, 6, 7, 9, 13]          [1, 2, 3, 4, 5]            5
  2     [3, 4, 11, 17, 19, 23]       [1, 2, 3, 5, 7, 8]         6
  3     [8, 12, 14, 18, 22, 24]      [1, 2, 3, 4, 6]            5
  4     [5, 10, 15, 21, 28, 30]      [1, 2, 4, 6, 7, 8, 9]      7
  5     [16, 20, 25, 26, 27, 29]     [1, 2, 3, 4, 5, 6, 9, 10]  8

  COSTE TOTAL: 31 actor-días
  Tiempo de ejecución: 0.005658388137817383 segundos


La técnica de diseño de Ramificación y Poda podría pensarse como una alternativa a descartar, sobre todo en problemas con un gran espacio de búsqueda, por ser una alternativa costosa que debería evaluar muchas soluciones.

Sin embargo, el establecimiento de un buen mecanismo de poda mediante la asignación de unas cotas superiores e inferiores adecuadas es clave en este problema, permitiendo llevar a cabo su implementación.

En este punto, el uso de esta técnica encaja bien debido a que previamente se ha usado un algoritmo voraz que, si bien no ha obtenido el óptimo global del problema, ha podido dar un resultado bueno que puede usarse como cota superior en el proceso de poda de las ramas que se generen.

Es decir, al partir de un resultado bueno obtenido con el algoritmo voraz, es viable y efectivo utilizar la Ramificación y Poda para obtener el óptimo global del problema. Esta forma de combinar técnicas, en la que un método rápido y aproximado guía o acelera a un método exacto, es una estrategia habitual en el campo de la optimización combinatoria y la inteligencia artificial, donde obtener una buena solución inicial es clave para reducir el espacio de búsqueda que el algoritmo exacto debe explorar.

Del resultado que se obtiene mediante el algoritmo voraz partirá el algoritmo de ramificación y poda, y se irá actualizando el valor del mejor coste acumulado conforme se vayan encontrando soluciones completas.

El mejor coste conocido en cada momento se utilizará en el método DFS para decidir si podar una rama. Para ello, antes de explorar cada nodo se calcula su cota inferior, siendo este el mínimo coste posible que puede alcanzar cualquier solución descendiente de ese nodo. Este cálculo comprueba en primer lugar si quedan tomas por asignar y, si todavía quedan, toma la toma con menor número de actores de entre las pendientes y multiplica su coste por el número mínimo de días que serán necesarios para grabar las tomas restantes, sumando el resultado al coste ya acumulado. Si esta cota inferior ya iguala o supera el mejor coste conocido, ningún descendiente de ese nodo puede mejorar la solución actual, por lo que la rama se poda.

In [7]:
def cota_inferior(indice_toma, coste_acumulado, orden_tomas, actores_por_toma) -> float:
    """Método que calcula la cota inferior a partir de un nodo."""

    tomas_restantes = n_tomas - indice_toma
    if tomas_restantes == 0:
        return coste_acumulado
    minimo_actores = min(
        actores_por_toma[orden_tomas[i]] for i in range(indice_toma, n_tomas)
    )
    dias_extra = math.ceil(tomas_restantes / MAX_TOMAS_DIA)
    return coste_acumulado + dias_extra * minimo_actores

Una vez se ha definido la forma de calcular el coste de la cota inferior y la poda de las ramas que no prometen soluciones mejores, se procede con la implementación del método de ramificación y poda. La búsqueda se realiza mediante DFS (búsqueda en profundidad), que explora cada rama del árbol hasta el final antes de retroceder y probar la siguiente:

1. El primer paso del algoritmo consiste en comprobar si se ha llegado al caso base, es decir, si ya se han asignado todas las tomas. Esto se realiza comprobando si el índice de la toma actual coincide con el número total de tomas (coincidiría al llegar al final debido al procesamiento iterativo e incremental). En caso de llegar al final, se comprueba si mejora el mejor coste hasta el momento, actualizando dicho valor de ser así.

2. El segundo paso consiste en el cálculo de la cota inferior de la forma que se comentó anteriormente.

3. Una vez calculada la cota inferior, se comprueba si su resultado es peor que el mejor caso, razón suficiente para parar el procesamiento por dicha rama del árbol (poda), mejorando así la eficiencia del algoritmo. Como desde un principio se parte del resultado obtenido con el algoritmo voraz, se parte de un resultado bastante bueno por lo que se podarán muchas ramas.

4. Si el coste de la cota inferior no supera el de la mejor solución, se procede con la generación de ramas a partir del nodo que se está procesando. Para cada nodo, las ramas posibles son añadir la toma a un día ya abierto o abrir un día nuevo. En el primer caso se calcula el coste marginal de cada día existente, es decir, cuántos actores nuevos incorporaría la toma. En el segundo, solo se considera abrir un nuevo día si no se supera el número mínimo de días necesarios más un margen de uno. En ambos casos se descartan días que tras la asignación tendrían el mismo conjunto de actores, ya que explorarlos sería redundante. Las ramas resultantes se ordenan por coste marginal creciente para explorar primero las asignaciones más baratas.

  Es necesario comprender, en este punto, el significado de cada parámetro usado:

  - num_tomas_por_dia: este parámetro es una lista que contiene el número de las tomas asignadas a cada día.
  - actores_por_dia: este parámetro es una lista de listas, es decir, una matriz, que contiene los actores convocados en cada día de la solución actual. En cada posición hay una lista que contiene valores binarios, tantos como actores hay, indicando si cada actor está convocado ese día o no. Esto permite, al generar las uniones con el método *np.maximum* con la fila de la matriz correspondiente a la toma actual, añadir los actores nuevos a los que ya estaban asignados para dicho día.

5. Por último, se explora cada rama generada llamando recursivamente al propio método DFS con el estado actualizado. Tras volver de cada llamada recursiva, se deshacen los cambios realizados para restaurar el estado anterior (backtracking), permitiendo así explorar la siguiente rama desde el mismo punto de partida.

In [8]:
def dfs(indice_toma, actores_por_dia, tomas_por_dia, num_tomas_por_dia, coste_acumulado, actores_por_toma, orden_tomas, dias_minimos, mejor_solucion):
    """Método principal que implementa ramificación y poda."""

    mejor_plan, mejor_coste = mejor_solucion

    # Caso base. Todas las tomas ya se han asignado
    if indice_toma == n_tomas:
        if coste_acumulado < mejor_coste[0]:
            mejor_coste[0] = coste_acumulado
            mejor_plan[0] = [list(s) for s in tomas_por_dia]
        return

    # Poda: cota inferior ≥ mejor coste conocido
    cota_inf = cota_inferior(indice_toma, coste_acumulado, orden_tomas, actores_por_toma)
    if cota_inf >= mejor_coste[0]:
        return

    toma = orden_tomas[indice_toma]

    # Generar ramas
    ramas = []
    uniones_vistas = set()

    for i in range(len(actores_por_dia)):
        if num_tomas_por_dia[i] < MAX_TOMAS_DIA:
            nueva_union = np.maximum(actores_por_dia[i], matriz[toma])
            clave = nueva_union.tobytes()
            if clave not in uniones_vistas:
                uniones_vistas.add(clave)
                coste_marginal = int(nueva_union.sum()) - int(actores_por_dia[i].sum())
                ramas.append((coste_marginal, i, nueva_union))

    # Se añade 1 como margen
    if len(actores_por_dia) < dias_minimos + 1:
        union_dia_nuevo = matriz[toma].copy().astype(int)
        clave = union_dia_nuevo.tobytes()
        if clave not in uniones_vistas:
            ramas.append((actores_por_toma[toma], -1, union_dia_nuevo))

    ramas.sort(key=lambda x: x[0])

    # Explorar ramas
    for coste_marginal, indice_dia, union_resultante in ramas:
        if indice_dia == -1: #Se abre un día nuevo
            actores_por_dia.append(union_resultante)
            tomas_por_dia.append([toma])
            num_tomas_por_dia.append(1)
            dfs(indice_toma + 1, actores_por_dia, tomas_por_dia, num_tomas_por_dia, coste_acumulado + coste_marginal, actores_por_toma, orden_tomas, dias_minimos, (mejor_plan, mejor_coste))
            actores_por_dia.pop()
            tomas_por_dia.pop()
            num_tomas_por_dia.pop()

        else: #Añadir a día existente
            union_anterior = actores_por_dia[indice_dia]
            actores_por_dia[indice_dia] = union_resultante
            tomas_por_dia[indice_dia].append(toma)
            num_tomas_por_dia[indice_dia] += 1
            dfs(indice_toma + 1, actores_por_dia, tomas_por_dia, num_tomas_por_dia, coste_acumulado + coste_marginal, actores_por_toma, orden_tomas, dias_minimos, (mejor_plan, mejor_coste))

            # Retroceso
            actores_por_dia[indice_dia] = union_anterior
            tomas_por_dia[indice_dia].pop()
            num_tomas_por_dia[indice_dia] -= 1

Una vez implementada toda la lógica del método de ramificación y poda, solo es necesario inicializar las estructuras necesarias. Se genera la lista de tomas ordenadas de mayor a menor por el número de actores que contiene y se precalcula una lista con el número de actores de cada toma para no tener que recalcularlo en cada nodo del árbol.

Partiendo del resultado del algoritmo voraz como cota superior inicial (coste y planificación), se llama al método DFS y se obtiene el mejor plan y coste encontrados para poder mostrarlos posteriormente.

In [9]:
def branch_and_bound(matriz, n_tomas, n_actores, max_tomas_dia, solucion_voraz) -> tuple:
    """Inicialización y llamada al método dfs"""

    orden_tomas    = sorted(range(n_tomas), key=lambda t: -int(matriz[t].sum()))
    actores_por_toma = [int(matriz[t].sum()) for t in range(n_tomas)]

    plan_inicial, coste_inicial = solucion_voraz
    mejor_coste = [coste_inicial]
    mejor_plan  = [plan_inicial]

    dias_minimos = math.ceil(n_tomas / MAX_TOMAS_DIA)
    dfs(0, [], [], [], 0, actores_por_toma, orden_tomas, dias_minimos, (mejor_plan, mejor_coste))

    return mejor_plan[0], mejor_coste[0]

In [10]:
start_time = time.time()
plan_optimo, coste_optimo = branch_and_bound(matriz, n_tomas, n_actores, MAX_TOMAS_DIA, solucion_voraz=(planificacionVoraz, costeVoraz))
end_time = time.time()

mostrar_resultado(plan_optimo, coste_optimo)
print(f"  Tiempo de ejecución: {end_time - start_time} segundos")

  Días necesarios : 5

  DÍA   TOMAS GRABADAS               ACTORES CONVOCADOS         COSTE
  --------------------------------------------------------------------
  1     [1, 5, 10, 11, 12, 26]       [1, 2, 3, 4, 5, 6, 8, 9]   8
  2     [3, 4, 8, 15, 21, 29]        [1, 2, 5, 6, 7, 8]         6
  3     [2, 6, 7, 13, 20, 22]        [1, 2, 3, 4, 5]            5
  4     [9, 16, 25, 27, 28, 30]      [1, 2, 4, 5, 10]           5
  5     [14, 17, 18, 19, 23, 24]     [1, 3, 6]                  3

  COSTE TOTAL: 27 actor-días
  Tiempo de ejecución: 9.872545957565308 segundos


# Análisis
- ¿Que complejidad tiene el problema? Orden de complejidad y contabilizar el espacio de soluciones

El problema de planificación de doblaje pertenece a la familia de problemas NP-difíciles, concretamente es una variante del problema de agrupamiento con restricciones, para el cual no se conoce ningún algoritmo de resolución exacta con complejidad polinomial. Esto significa que el tiempo necesario para garantizar la solución óptima crece de forma exponencial con el tamaño de la entrada en el caso general.

El espacio de soluciones, es decir, el número de formas posibles de asignar las 30 tomas a 5 días sin tener en cuenta ninguna restricción, puede estimarse como el número de particiones de 30 elementos en 5 grupos, que es del orden de 5^30 soluciones posibles. Explorar exhaustivamente este espacio es inviable en la práctica, lo que justifica el uso de técnicas como el algoritmo voraz y la ramificación y poda.

En primer lugar, es necesario indicar que el algoritmo voraz tiene el funcionamiento esperado en este tipo de resolución, pues obtiene un resultado aproximado al óptimo del problema en bajos tiempos de ejecución. El algoritmo de ramificación y poda obtiene también bajos tiempos de ejecución, y esto se debe a que se utiliza el resultado del algoritmo voraz como cota superior inicial, permitiendo podar muchas ramas desde el principio. Sin esta combinación de técnicas, la ramificación y poda seguiría siendo eficaz, pues obtendría igualmente el óptimo global, pero sería menos eficiente, pues tardaría bastante más en encontrarlo al tener que recorrer muchas más ramas.

Para estudiar el orden de complejidad será necesario comprobar el número de operaciones elementales que realizan los algoritmos, dando el resultado en notación Big O. Para el algoritmo voraz, los dos métodos que se implementaron para calcular el coste de una sesión y los actores de una sesión, *coste_sesion* y *actores_en_sesion* respectivamente, tienen complejidad O(n_tomas · n_actores), pues internamente recorren las tomas de una sesión y por cada una operan sobre un vector de tamaño n_actores.

El algoritmo *algoritmo_voraz* se ha implementado de forma iterativa, no recursiva, por lo que el problema de espacio en memoria que ocurre con las soluciones recursivas no se presenta. Internamente, este algoritmo recorre las tomas pendientes y, para cada día, vuelve a recorrer las tomas pendientes para seleccionar la mejor asignación, realizando en cada comparación una operación sobre vectores de tamaño n_actores. Debido a esto, el orden de complejidad es O(n_tomas² · n_actores), que tratando n_actores como constante se puede expresar como O(n²).

Con respecto a la solución mediante ramificación y poda, el método de cálculo de la cota inferior realiza un bucle en función del número de tomas, lo que hace que su complejidad sea lineal, O(n). El algoritmo *dfs* hace uso del cálculo de cotas (n operaciones) y la generación de ramas (n operaciones) y, por último, la exploración de ramas mediante un bucle y recursividad. Su complejidad no es fija ni expresable en notación Big O de forma cerrada, ya que depende directamente de la efectividad de la poda. En el peor caso, sin ninguna poda, la complejidad sería O(n_dias^n_tomas), equivalente a una búsqueda exhaustiva. En el mejor caso, si la cota superior inicial es suficientemente buena para podar todas las ramas salvo la óptima, la complejidad se reduciría a O(n_tomas). El comportamiento real se sitúa entre ambos extremos y depende directamente de la calidad de la cota superior proporcionada por el algoritmo voraz, que en este problema permite reducir el número de nodos explorados de forma muy significativa.

*Nota sobre Inteligencia Artificial*

Se ha usado la IA de Google Gemini para:

1. Una vez realizado un borrador, mejorar la redacción y el estilo de la explicación del código.
2. Solucionar un error de ejecución en la generación de ramas. Para ello se le ha pasado el error de ejecución, el código anterior que lo provocaba, y ha propuesto la solución que se observa en el código final.